## Google Colab and data paths

Set `KDD_PROJECT_ROOT` if your project is not stored at `/content/drive/MyDrive/kdd_dropout_project`. The configured folder must contain `dataset/`, `shared/metadata/`, and `artifacts/`.


In [ ]:
from pathlib import Path
import os

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    pass

def discover_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset").exists() and (candidate / "shared" / "metadata").exists():
            return candidate
    return start

default_root = (
    Path("/content/drive/MyDrive/kdd_dropout_project")
    if IN_COLAB else discover_project_root(Path.cwd())
)
PROJECT_ROOT = Path(os.environ.get("KDD_PROJECT_ROOT", default_root))
DATA_ROOT = PROJECT_ROOT / "dataset"
METADATA_ROOT = PROJECT_ROOT / "shared" / "metadata"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
MODEL_ARTIFACT_ROOT = ARTIFACT_ROOT / "feature_engineering"
MODEL_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Model artifacts: {MODEL_ARTIFACT_ROOT}")


# FSLSM — All Four Dimensions, Binary Scoring

This extends the `Binary_Visual_Verbal_Scoring.pdf` methodology (a difference-based ratio
score plus a binary tie-broken label) from the **Input (Visual/Verbal)** dimension to the
other three dimensions in the `KDD Cup 2015 -> FSLSM Mapping` table: **Processing
(Active/Reflective)**, **Perception (Sensing/Intuitive)**, and **Understanding
(Sequential/Global)**.

**Important — read before using downstream:** the PDF gives an exact, unambiguous formula
for Input because both poles reduce to raw event counts (`video+access` vs `wiki+discussion`).
The mapping table's other three rows are not fully specified that way — several poles carry
qualifiers (*"high dwell"*, *"high retry"*, *"broad"*, *"low variance"*, *"jumps"*) that
describe a **behavioral pattern**, not a literal event count, and the table doesn't define the
thresholds. Below, each qualifier is operationalized explicitly and documented; these are
reasonable, defensible choices consistent with the FSLSM literature's intent, but they are
**assumptions this notebook is making on your behalf, not settled definitions from your
source documents** — flag them to whoever is grading/supervising this before treating the
Processing/Perception/Understanding scores as equally authoritative as the Input score.

All four dimensions follow the identical pattern from the PDF:

```
Score_i = (Pole_A_i - Pole_B_i) / (Pole_A_i + Pole_B_i)      in [-1, 1]
Label_i = PoleA   if Pole_A_i > Pole_B_i
        = PoleB   if Pole_A_i < Pole_B_i
        = PoleB   if Pole_A_i == Pole_B_i   (tie-break rule, consistent with the PDF's
                                             "ties -> Verbal" convention: ties go to the
                                             second-listed pole in every dimension)
```

| Dimension     | Pole A (→ tie-break away from) | Pole B (→ tie-break target) |
|---|---|---|
| Input         | Visual                         | **Verbal**                  |
| Processing    | Active                         | **Reflective**               |
| Perception    | Sensing                        | **Intuitive**                |
| Understanding | Sequential                     | **Global**                   |

## 1. Load data & shared preprocessing

Same first-2-weeks filter and category merge as `02_Data_Preprocessing.ipynb`, so these
scores describe the same window of behavior as the sequence models.

In [ ]:
import pandas as pd
import numpy as np
import os

print("Loading raw data...")
log_train = pd.read_csv(DATA_ROOT / "train/train/log_train.csv")
log_test = pd.read_csv(DATA_ROOT / "test/test/log_test.csv")
object_df = pd.read_csv(METADATA_ROOT / "object.csv")

log_train['time'] = pd.to_datetime(log_train['time'])
log_test['time'] = pd.to_datetime(log_test['time'])

def filter_first_2_weeks(df):
    first_time = df.groupby("enrollment_id")["time"].transform("min")
    return df[(df["time"] - first_time).dt.total_seconds() <= 14 * 24 * 3600].copy()

log_train = filter_first_2_weeks(log_train)
log_test = filter_first_2_weeks(log_test)

log_train = log_train.merge(object_df[['module_id', 'category']], how='left', left_on='object', right_on='module_id')
log_test = log_test.merge(object_df[['module_id', 'category']], how='left', left_on='object', right_on='module_id')
log_train['category'] = log_train['category'].fillna("unknown")
log_test['category'] = log_test['category'].fillna("unknown")

# Raw (unlogged) time gap between a learner's consecutive events, in seconds — needed for
# the dwell-time and navigation-variance operationalizations below. Sorting first is required
# for .diff() to mean "time since this learner's previous event".
def add_raw_time_gap(df):
    df = df.sort_values(['enrollment_id', 'time']).copy()
    df['time_gap_sec'] = df.groupby('enrollment_id')['time'].diff().dt.total_seconds().fillna(0)
    return df

log_train = add_raw_time_gap(log_train)
log_test = add_raw_time_gap(log_test)

print(f"log_train: {log_train.shape}, log_test: {log_test.shape}")

## 2. Input dimension — Visual / Verbal

**Unchanged from the PDF** — the one dimension with an exact, literal-count formula.
`V = video + access`, `R = wiki + discussion`.

In [ ]:
def compute_input_dimension(df):
    # V_i = video_i + access_i ; R_i = wiki_i + discussion_i  (verbatim from the PDF)
    counts = df.groupby(['enrollment_id', 'event']).size().unstack(fill_value=0)
    for col in ['video', 'access', 'wiki', 'discussion']:
        if col not in counts.columns:
            counts[col] = 0

    V = counts['video'] + counts['access']
    R = counts['wiki'] + counts['discussion']
    total = V + R
    S = np.where(total > 0, (V - R) / total, 0.0)
    label = np.where(V > R, 'Visual', 'Verbal')  # ties -> Verbal

    return pd.DataFrame({
        'student_id': counts.index,
        'input_visual_count': V.values,
        'input_verbal_count': R.values,
        'input_S': S,
        'input_label': label,
    }).reset_index(drop=True)

## 3. Processing dimension — Active / Reflective

Table: *Active* = `problem, discussion` (plain counts — no qualifier, so this is literal,
just like Input). *Reflective* = `video, access` **with a "high dwell" qualifier** — the
table is explicitly distinguishing this from Input's Visual pole (which uses the *same two
events*) by how long the learner lingers, not how often they click.

**Operationalization:** rather than counting video/access events, sum the raw time spent
immediately before/after them — i.e. use each event's `time_gap_sec` as a proxy for dwell
time on the material, and sum it across video/access events. This turns "high dwell" from a
count into a duration, which is the only way to make Reflective meaningfully different from
Input's Visual pole given they share the same two events. `Active` stays a plain count, so
the two poles are on different underlying units (an event count vs. a summed duration) —
worth flagging when interpreting `processing_S` numerically.

In [ ]:
def compute_processing_dimension(df):
    # Active_i = count(problem) + count(discussion)  — literal, per the table.
    active_counts = df[df['event'].isin(['problem', 'discussion'])].groupby('enrollment_id').size()

    # Reflective_i = total dwell time (sum of time_gap_sec) accumulated on video/access
    # events — an operationalization of "high dwell", since the table gives no threshold.
    reflective_dwell = (
        df[df['event'].isin(['video', 'access'])]
        .groupby('enrollment_id')['time_gap_sec'].sum()
    )

    all_ids = df['enrollment_id'].unique()
    out = pd.DataFrame({'student_id': all_ids}).set_index('student_id')
    out['processing_active_count'] = active_counts
    out['processing_reflective_dwell'] = reflective_dwell
    out = out.fillna(0.0)

    A, Rf = out['processing_active_count'], out['processing_reflective_dwell']
    total = A + Rf
    S = np.where(total > 0, (A - Rf) / total, 0.0)
    label = np.where(A > Rf, 'Active', 'Reflective')  # ties -> Reflective

    out = out.reset_index()
    out['processing_S'] = S
    out['processing_label'] = label
    return out[['student_id', 'processing_active_count', 'processing_reflective_dwell', 'processing_S', 'processing_label']]

## 4. Perception dimension — Sensing / Intuitive

Table: *Sensing* = `problem` **with a "high retry" qualifier** (concrete, trial-and-error
learners re-attempt the same problem). *Intuitive* = `wiki, navigate` **with a "broad"
qualifier** (abstract learners range widely rather than drilling down).

**Operationalization:**
- `Sensing_i` = retry count = for every distinct object a learner sees a `problem` event on,
  count repeat visits beyond the first, summed across objects. A learner who hits the same
  problem 4 times contributes 3 retries; a learner who never repeats contributes 0 regardless
  of total problem count — this is what "retry" means as distinct from a plain problem count.
- `Intuitive_i` = `count(wiki)` + the number of **distinct objects** reached via `navigate`
  events (breadth of exploration) — "broad" is operationalized as touching many different
  places rather than looping on one.

In [ ]:
def compute_perception_dimension(df):
    # Sensing_i: retries on 'problem' events, i.e. repeat visits to the SAME object beyond
    # the first attempt, summed across objects for that learner.
    problem_df = df[df['event'] == 'problem']
    visits_per_object = problem_df.groupby(['enrollment_id', 'object']).size()
    retries_per_object = (visits_per_object - 1).clip(lower=0)
    sensing_retries = retries_per_object.groupby('enrollment_id').sum()

    # Intuitive_i: count(wiki) + breadth of navigate exploration (# distinct objects reached
    # via 'navigate').
    wiki_counts = df[df['event'] == 'wiki'].groupby('enrollment_id').size()
    navigate_breadth = df[df['event'] == 'navigate'].groupby('enrollment_id')['object'].nunique()

    all_ids = df['enrollment_id'].unique()
    out = pd.DataFrame({'student_id': all_ids}).set_index('student_id')
    out['perception_sensing_retries'] = sensing_retries
    out['perception_intuitive_wiki'] = wiki_counts
    out['perception_intuitive_breadth'] = navigate_breadth
    out = out.fillna(0.0)
    out['perception_intuitive_score'] = out['perception_intuitive_wiki'] + out['perception_intuitive_breadth']

    Sn, In = out['perception_sensing_retries'], out['perception_intuitive_score']
    total = Sn + In
    S = np.where(total > 0, (Sn - In) / total, 0.0)
    label = np.where(Sn > In, 'Sensing', 'Intuitive')  # ties -> Intuitive

    out = out.reset_index()
    out['perception_S'] = S
    out['perception_label'] = label
    return out[['student_id', 'perception_sensing_retries', 'perception_intuitive_score', 'perception_S', 'perception_label']]

## 5. Understanding dimension — Sequential / Global

Table: both poles map to the *same* `navigate` event and the *same* object categories
(`chapter, sequential, vertical`) — the only distinguishing signal is the qualifier:
*Sequential* = `navigate (low variance)`; *Global* = `navigate (jumps)` + `page_close`.

**Operationalization:** for each learner, look at the time gaps between their consecutive
`navigate` events. A "jump" is a gap that is a statistical outlier *relative to that
learner's own navigation rhythm* (Tukey's rule: `gap > Q3 + 1.5*IQR` of their own navigate
gaps) — using a per-learner threshold rather than one fixed cutoff avoids penalizing
naturally slower- or faster-paced learners.

- `Sequential_i` = count(navigate) − jump_count_i (the steady, non-jump transitions)
- `Global_i` = jump_count_i + count(page_close)

Learners with 3 or fewer navigate events don't have enough gaps to estimate an IQR reliably,
so their jump_count defaults to 0 (all their navigation counts as "sequential" by default) —
this is a conservative choice worth flagging, since it slightly favors the Sequential label
for low-activity learners.

In [ ]:
def _count_jumps(gaps):
    gaps = np.asarray(gaps)
    if len(gaps) < 4:
        return 0
    q1, q3 = np.percentile(gaps, [25, 75])
    iqr = q3 - q1
    threshold = q3 + 1.5 * iqr
    return int((gaps > threshold).sum())


def compute_understanding_dimension(df):
    nav_df = df[df['event'] == 'navigate'].sort_values(['enrollment_id', 'time']).copy()
    nav_df['nav_gap'] = nav_df.groupby('enrollment_id')['time'].diff().dt.total_seconds().fillna(0)

    nav_counts = nav_df.groupby('enrollment_id').size()
    jump_counts = nav_df.groupby('enrollment_id')['nav_gap'].apply(_count_jumps)
    page_close_counts = df[df['event'] == 'page_close'].groupby('enrollment_id').size()

    all_ids = df['enrollment_id'].unique()
    out = pd.DataFrame({'student_id': all_ids}).set_index('student_id')
    out['understanding_navigate_total'] = nav_counts
    out['understanding_jump_count'] = jump_counts
    out['understanding_page_close'] = page_close_counts
    out = out.fillna(0.0)

    out['understanding_sequential_score'] = (out['understanding_navigate_total'] - out['understanding_jump_count']).clip(lower=0)
    out['understanding_global_score'] = out['understanding_jump_count'] + out['understanding_page_close']

    Sq, Gl = out['understanding_sequential_score'], out['understanding_global_score']
    total = Sq + Gl
    S = np.where(total > 0, (Sq - Gl) / total, 0.0)
    label = np.where(Sq > Gl, 'Sequential', 'Global')  # ties -> Global

    out = out.reset_index()
    out['understanding_S'] = S
    out['understanding_label'] = label
    return out[['student_id', 'understanding_sequential_score', 'understanding_global_score', 'understanding_S', 'understanding_label']]

## 6. Combine all four dimensions & save

Produces one row per learner with all four `S` scores and binary labels — a drop-in
replacement/superset of the earlier `fslsm_train.csv` / `fslsm_test.csv` (which only had
the Input/Visual-Verbal columns). Downstream models can use just the four `_S` columns
(continuous, already in [-1, 1], no scaling needed) plus the four `_label` columns
(recommended: map to 0/1 and optionally one-hot), or the richer intermediate count columns
if a model wants raw signal instead of the compressed ratio.

In [ ]:
def compute_all_fslsm_dimensions(df):
    input_df = compute_input_dimension(df)
    processing_df = compute_processing_dimension(df)
    perception_df = compute_perception_dimension(df)
    understanding_df = compute_understanding_dimension(df)

    out = input_df.merge(processing_df, on='student_id', how='outer') \
                   .merge(perception_df, on='student_id', how='outer') \
                   .merge(understanding_df, on='student_id', how='outer')
    return out

fslsm_train_full = compute_all_fslsm_dimensions(log_train)
fslsm_test_full = compute_all_fslsm_dimensions(log_test)

MODEL_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
fslsm_train_full.to_csv(MODEL_ARTIFACT_ROOT / "fslsm_full_train.csv", index=False)
fslsm_test_full.to_csv(MODEL_ARTIFACT_ROOT / "fslsm_full_test.csv", index=False)

print(f"fslsm_train_full: {fslsm_train_full.shape}")
print(fslsm_train_full.head())
print("\nLabel distributions (train):")
for dim in ['input', 'processing', 'perception', 'understanding']:
    print(f"  {dim}: {fslsm_train_full[f'{dim}_label'].value_counts().to_dict()}")

s_cols = ['input_S', 'processing_S', 'perception_S', 'understanding_S']
print("\nPairwise correlation between the four dimension scores (sanity check —")
print("if these are near +-1, two 'dimensions' are barely distinguishable from each other):")
print(fslsm_train_full[s_cols].corr().round(3))

print(f"\nSaved: {MODEL_ARTIFACT_ROOT / 'fslsm_full_train.csv'}, {MODEL_ARTIFACT_ROOT / 'fslsm_full_test.csv'}")

## Summary of assumptions made (flag these before treating results as final)

| Dimension | Table's qualifier | What this notebook computed | Why |
|---|---|---|---|
| Input | *(none — literal counts)* | `video+access` vs `wiki+discussion` | Exact PDF formula, no assumption needed. |
| Processing | Reflective: "high dwell" | Summed `time_gap_sec` on video/access events, vs. a plain count for Active | No dwell threshold was given; using duration directly avoids picking an arbitrary cutoff, but means the two poles are on different units (count vs. seconds). |
| Perception | Sensing: "high retry"; Intuitive: "broad" | Retry = repeat visits to the same object beyond the first; breadth = distinct objects reached via navigate | No retry threshold given; "beyond the first visit" is the most literal reading of "retry". |
| Understanding | Sequential: "low variance"; Global: "jumps" | Per-learner Tukey-outlier navigate gaps count as jumps; non-jump navigates are Sequential | No variance/jump threshold given; a per-learner (not global) outlier rule avoids penalizing naturally slow/fast learners, at the cost of needing >=4 navigate events to estimate reliably. |

None of these three extended dimensions carry the same evidentiary weight as the PDF's
Input formula — they're reasonable operational choices, not verified definitions. If this
is being graded or reviewed, it's worth calling out explicitly that Input is "per spec" and
the other three are "team-defined operationalizations of the mapping table.\"